In [14]:
import os
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from skimage.feature import hog

# Transfer learning approach: extract HOG (Histogram of Oriented Gradients) features
# which capture edge/texture information, then train a lightweight classifier


In [5]:
healthy_classes = [
    "Apple", "Avocado", "Banana", "Beef", "Beef-Fillet", "Beef-Goulash", "Beef-Patty",
    "Blueberry-Yogurt", "Grapes", "Grilled-Chicken-Salad", "Grilled-Pizza", "Ground-Beef",
    "Honey", "Jackfruit", "Kiwi", "Lemon", "Lime", "Lychees", "Mango", "Milk", "Mint",
    "Mozzarella", "Orange", "Pancake", "Papaya", "Parsley", "Passion-Fruit", "Peach",
    "Pear", "Pineapple", "Plum", "Raspberries", "Sandwich", "Strawberries", "Tangerine",
    "Watermelon", "Yogurt", "American-Cheese"
]

unhealthy_classes = [
    "BBQ-Chicken-Pizza", "BBQ-Pizza", "Beef-Pizza", "Beef-Ribs", "Beef-Steak", "Biscuit",
    "Cheese-Pizza", "Chicken-Pizza", "Cupcake", "Donut", "Double-Cheeseburger", "Egg-Roll",
    "Hot-Chocoloate", "Hot-Dog", "Mayonnaise", "Muffin", "Mustard", "Pie", "Sausage-Pizza",
    "Seafood-Pizza", "Vanilla-Yogurt", "Vegetable-Pizza", "Vegetarian-Pizza", "Waffles",
    "Zinger-Burger", "Chocolate-Yogurt", "Strawberry-Jam"
]

In [6]:
train_dir = r"c:\Users\stangutur\DS-4002-Group4Project3\DATA\Nutrition Food Proj_Cleaned\Nutrition Food Proj -B0-.folder (1)\train"

X = []
y = []

IMG_SIZE = 224  # Image size for consistent input

for class_folder in os.listdir(train_dir):
    class_path = os.path.join(train_dir, class_folder)

    # skip non-folders
    if not os.path.isdir(class_path):
        continue

    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)

        # Assign label
        if class_folder in healthy_classes:
            label = 1
        elif class_folder in unhealthy_classes:
            label = 0
        else:
            continue  # skip unknown classes

        # Load and preprocess image
        img = cv2.imread(img_path)
        if img is None:
            continue  # skip unreadable images

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

        X.append(img)
        y.append(label)

X = np.array(X)
y = np.array(y)

print("Loaded images:", X.shape)
print("Loaded labels:", y.shape)

Loaded images: (653, 224, 224, 3)
Loaded labels: (653,)


In [ ]:
# ---------------------------------------------------------
# 4. TRANSFER LEARNING FEATURE EXTRACTION (HOG)
# ---------------------------------------------------------
# Using HOG (Histogram of Oriented Gradients): robust, effective features for image classification


In [8]:
# ---------------------------------------------------------
# 3. TRAIN/TEST SPLIT
# ---------------------------------------------------------

print("Splitting data into train and test sets...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Splitting data into train and test sets...
Training set size: 522
Test set size: 131


In [15]:
# ---------------------------------------------------------
# 5. EXTRACT HOG FEATURES AND TRAIN CLASSIFIER
# ---------------------------------------------------------
def extract_hog_features(images):
    """Extract HOG (Histogram of Oriented Gradients) features."""
    features = []
    for img in images:
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        # Extract HOG features
        hog_features = hog(gray, orientations=9, pixels_per_cell=(8, 8), 
                           cells_per_block=(2, 2), visualize=False)
        features.append(hog_features)
    return np.array(features)

print("Extracting HOG features from training set...")
X_train_hog = extract_hog_features(X_train)
print(f"Training HOG features shape: {X_train_hog.shape}")

print("Extracting HOG features from test set...")
X_test_hog = extract_hog_features(X_test)
print(f"Test HOG features shape: {X_test_hog.shape}")

print("Training Logistic Regression on HOG features...")
clf = LogisticRegression(max_iter=2000)
clf.fit(X_train_hog, y_train)
print("Model training completed!")

y_pred = clf.predict(X_test_hog)
print("=== TRANSFER LEARNING RESULTS (HOG + Logistic Regression) ===")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Extracting HOG features from training set...
Training HOG features shape: (522, 26244)
Extracting HOG features from test set...
Test HOG features shape: (131, 26244)
Training Logistic Regression on HOG features...
Model training completed!
=== TRANSFER LEARNING RESULTS (HOG + Logistic Regression) ===
Accuracy: 0.5190839694656488

Confusion Matrix:
 [[17 37]
 [26 51]]

Classification Report:
               precision    recall  f1-score   support

           0       0.40      0.31      0.35        54
           1       0.58      0.66      0.62        77

    accuracy                           0.52       131
   macro avg       0.49      0.49      0.48       131
weighted avg       0.50      0.52      0.51       131

